## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-05-14-51-23 +0000,nypost,Bizarre moment student sent flying by runaway ...,https://nypost.com/2026/11/05/world-news/chile...
1,2026-11-05-14-51-07 +0000,nyt,Bangladesh Agrees to Raise Minimum Wage for Ga...,https://www.nytimes.com/2026/11/05/world/asia/...
2,2026-11-05-14-49-49 +0000,nypost,Death toll rises to 5 in Kenya building collap...,https://nypost.com/2026/11/05/world-news/death...
3,2026-11-05-14-48-39 +0000,nypost,Bomb hoaxes grounding 25 planes include bizarr...,https://nypost.com/2026/11/05/world-news/bomb-...
4,2026-11-05-14-48-00 +0000,wsj,Drone Startup Zipline Raises $1.5 Billion at $...,https://www.wsj.com/business/drone-startup-zip...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2467/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
59,trump,55
66,midterm,36
139,democrats,33
142,midterms,26
326,war,21


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
357,2026-11-04-20-06-00 +0000,wsj,"Democrats won control of Congress, a majority ...",https://www.wsj.com/politics/elections/how-dem...,207
399,2026-11-04-18-06-00 +0000,wsj,"To win control of the House and Senate, Democr...",https://www.wsj.com/politics/elections/the-vot...,179
286,2026-11-04-22-37-00 +0000,wsj,Democrats’ midterm sweep has shaken up the 202...,https://www.wsj.com/politics/elections/democra...,178
418,2026-11-04-16-55-29 +0000,chitri,Iran’s leaders say midterm results show US vot...,https://www.chicagotribune.com/2026/11/04/iran...,165
298,2026-11-04-22-12-56 +0000,nypost,US withdrawing $4B in Israel aid decided month...,https://nypost.com/2026/11/04/us-news/us-to-wi...,164


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
357,207,2026-11-04-20-06-00 +0000,wsj,"Democrats won control of Congress, a majority ...",https://www.wsj.com/politics/elections/how-dem...
290,117,2026-11-04-22-31-00 +0000,wsj,Vice President JD Vance blamed the war in Iran...,https://www.wsj.com/politics/policy/vances-bre...
286,106,2026-11-04-22-37-00 +0000,wsj,Democrats’ midterm sweep has shaken up the 202...,https://www.wsj.com/politics/elections/democra...
243,86,2026-11-05-00-20-57 +0000,chitri,Israel’s president takes first steps on buildi...,https://www.chicagotribune.com/2026/11/04/isra...
399,81,2026-11-04-18-06-00 +0000,wsj,"To win control of the House and Senate, Democr...",https://www.wsj.com/politics/elections/the-vot...
324,67,2026-11-04-21-21-00 +0000,wsj,Markets surged Wednesday after American voters...,https://www.wsj.com/politics/elections/the-win...
348,61,2026-11-04-20-23-48 +0000,nypost,Jane Kim holds slim lead over Ben Allen with r...,https://nypost.com/2026/11/04/us-news/jane-kim...
227,57,2026-11-05-01-02-54 +0000,nypost,GOP Rep. Nick LaLota claims victory against ‘k...,https://nypost.com/2026/11/04/us-news/gop-rep-...
51,56,2026-11-05-13-26-00 +0000,wsj,The Bank of England raised its key interest ra...,https://www.wsj.com/economy/central-banking/ba...
242,50,2026-11-05-00-26-00 +0000,nypost,NY Rep. Mike Lawler loses re-election in blue ...,https://nypost.com/2026/11/04/us-news/lawler-l...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
